In [1]:
import os
os.chdir('..')
import sys

os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'
set_gpus(1, forcing=True)
print(f"XLA_PYTHON_CLIENT_ALLOCATOR set to: {os.environ.get('XLA_PYTHON_CLIENT_ALLOCATOR')}")
print(f"CUDA_VISIBLE_DEVICES set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

XLA_PYTHON_CLIENT_ALLOCATOR set to: platform
CUDA_VISIBLE_DEVICES set to: 1


In [2]:
from data import DL17Lands

In [3]:
dl = DL17Lands('WOE')

Loading WOE
-----------
Loading card data: 0.065368s
Making extension dataframe: 0.535147s
##################################################


In [4]:
vars(dl).keys()

dict_keys(['format', 'all_cards', 'verbose', 'cards', 'drafts', 'pack_size'])

In [5]:
dl.drafts.collect_schema().names()

['expansion',
 'draft_id',
 'draft_time',
 'rank',
 'event_match_wins',
 'event_match_losses',
 'pack_number',
 'pick_number',
 'pick',
 'pick_maindeck_rate',
 'pick_sideboard_in_rate',
 'user_n_games_bucket',
 'user_game_win_rate_bucket',
 'pick_id',
 'pool',
 'pack']

In [6]:
import polars as pl

dl_tmp = dl.drafts.group_by('draft_id').agg(pl.col('event_match_wins').mean(), pl.col('event_match_losses').mean())
dl_tmp.select(pl.col('event_match_wins').unique()).collect(), dl_tmp.select(pl.col('event_match_losses').unique()).collect()

(shape: (8, 1)
 ┌──────────────────┐
 │ event_match_wins │
 │ ---              │
 │ f64              │
 ╞══════════════════╡
 │ 0.0              │
 │ 1.0              │
 │ 2.0              │
 │ 3.0              │
 │ 4.0              │
 │ 5.0              │
 │ 6.0              │
 │ 7.0              │
 └──────────────────┘,
 shape: (4, 1)
 ┌────────────────────┐
 │ event_match_losses │
 │ ---                │
 │ f64                │
 ╞════════════════════╡
 │ 0.0                │
 │ 1.0                │
 │ 2.0                │
 │ 3.0                │
 └────────────────────┘)

In [7]:
stats = dl_tmp.select(pl.col('event_match_wins').sum(), pl.col('event_match_losses').sum()).collect()
stats, stats['event_match_wins'][0] / (stats['event_match_wins'][0] + stats['event_match_losses'][0])

(shape: (1, 2)
 ┌──────────────────┬────────────────────┐
 │ event_match_wins ┆ event_match_losses │
 │ ---              ┆ ---                │
 │ f64              ┆ f64                │
 ╞══════════════════╪════════════════════╡
 │ 407191.0         ┆ 350020.0           │
 └──────────────────┴────────────────────┘,
 0.5377510363689909)